In [ ]:
# =========================================================
# Lightweight Alignment of Vision-Language Representations
# via 1D U-Net Residual Embedding Denoising (CLIP)
# =========================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoProcessor, AutoModelForZeroShotImageClassification
from datasets import load_dataset
import matplotlib.pyplot as plt
import numpy as np

# =========================================================
# 1️Define lightweight 1D U-Net Residual Denoiser
# =========================================================

class UNet1DResidualDenoiser(nn.Module):
    """
    1D U-Net residual denoiser for CLIP embeddings.
    Formula: x_denoised = x_noisy - UNet1D(x_noisy - x_clean)
    """
    def __init__(self, embedding_dim=512, hidden_dim=1024):
        super().__init__()

        # Encoder
        self.enc1 = nn.Conv1d(embedding_dim, hidden_dim, kernel_size=1)
        self.enc2 = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=1)

        # Decoder
        self.dec1 = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=1)
        self.dec2 = nn.Conv1d(hidden_dim, embedding_dim, kernel_size=1)

        self.act = nn.ReLU()

    def forward(self, x_noisy, x_clean):
        # Compute residual noise
        residual = x_noisy - x_clean          # ε = x_noisy - x_clean
        residual = residual.unsqueeze(-1)     # [batch, embedding_dim, 1] for Conv1D

        # Encoder
        e1 = self.act(self.enc1(residual))
        e2 = self.act(self.enc2(e1))

        # Decoder with skip connection
        d1 = self.act(self.dec1(e2) + e1)
        d2 = self.dec2(d1).squeeze(-1)        # [batch, embedding_dim]

        # Residual subtraction
        x_denoised = x_noisy - d2
        return x_denoised

# =========================================================
# 2️⃣ Helper: Add Gaussian noise to embeddings
# =========================================================

def add_noise(embeddings, noise_std=0.2):
    return embeddings + torch.randn_like(embeddings) * noise_std

# =========================================================
# 3️⃣ Load dataset (small subset for demo)
# =========================================================

dataset = load_dataset("SKyu/my-image-captioning-dataset")
train_dataset = dataset["train"].select(range(2000, 3000))
val_dataset = dataset["train"].select(range(1, 1000))

def transformFn(batch):
    images = [item["image"] for item in batch]
    captions = [item["prompt"] for item in batch]
    return images, captions

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=transformFn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=transformFn)

# =========================================================
# 4️⃣ Load frozen CLIP model
# =========================================================

device = "cpu"

processor = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model = AutoModelForZeroShotImageClassification.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)

# Freeze CLIP parameters
clip_model.eval()
for param in clip_model.parameters():
    param.requires_grad = False

# =========================================================
# 5️⃣ Initialize 1D U-Net denoiser
# =========================================================

denoiser = UNet1DResidualDenoiser(embedding_dim=512, hidden_dim=1024).to(device)
optimizer = torch.optim.Adam(denoiser.parameters(), lr=1e-4)
criterion = nn.MSELoss()

# =========================================================
# 6️⃣ Training loop
# =========================================================

num_epochs = 10

for epoch in range(num_epochs):
    denoiser.train()
    epoch_loss = 0
    for images, captions in train_loader:
        # Prepare inputs
        inputs = processor(text=captions, images=images, return_tensors="pt",
                           padding=True, truncation=True, max_length=77).to(device)

        with torch.no_grad():
            # Get normalized CLIP embeddings
            img_embeds = F.normalize(clip_model.get_image_features(inputs["pixel_values"]), dim=-1)
            text_inputs = {k: v for k, v in inputs.items() if k in ["input_ids", "attention_mask"]}
            txt_embeds = F.normalize(clip_model.get_text_features(**text_inputs), dim=-1)

        # Add noise
        noisy_img = add_noise(img_embeds)
        noisy_txt = add_noise(txt_embeds)

        # Denoiser forward
        denoised_img = denoiser(noisy_img, img_embeds)
        denoised_txt = denoiser(noisy_txt, txt_embeds)

        # Loss with optional weighting λ1, λ2
        λ1, λ2 = 1.0, 1.0
        loss = λ1 * criterion(denoised_img, img_embeds) + λ2 * criterion(denoised_txt, txt_embeds)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Avg Loss: {avg_loss:.6f}")

print("✅ Residual denoiser training complete!")

# =========================================================
# Embedding extraction functions
# =========================================================

def get_raw_embeddings(images, captions):
    clip_model.eval()
    with torch.no_grad():
        inputs = processor(text=captions, images=images, return_tensors="pt",
                           padding=True, truncation=True, max_length=77).to(device)
        img_embeds = F.normalize(clip_model.get_image_features(inputs["pixel_values"]), dim=-1)
        text_inputs = {k: v for k, v in inputs.items() if k in ["input_ids", "attention_mask"]}
        txt_embeds = F.normalize(clip_model.get_text_features(**text_inputs), dim=-1)
    return img_embeds, txt_embeds

def get_denoised_embeddings(images, captions):
    denoiser.eval()
    img_embeds, txt_embeds = get_raw_embeddings(images, captions)
    denoised_img = F.normalize(denoiser(img_embeds, img_embeds), dim=-1)
    denoised_txt = F.normalize(denoiser(txt_embeds, txt_embeds), dim=-1)
    return denoised_img, denoised_txt

# =========================================================
# Evaluation function (Recall@K)
# =========================================================

def evaluate(loader, use_denoiser=False):
    recall_at_1, recall_at_3, recall_at_5, total = 0, 0, 0, 0

    for images, captions in loader:
        if use_denoiser:
            img_embeds, txt_embeds = get_denoised_embeddings(images, captions)
        else:
            img_embeds, txt_embeds = get_raw_embeddings(images, captions)

        sim_matrix = torch.matmul(img_embeds, txt_embeds.T)
        for i in range(sim_matrix.size(0)):
            scores = sim_matrix[i]
            top1 = scores.topk(min(1, scores.size(0))).indices[0].item()
            top3 = scores.topk(min(3, scores.size(0))).indices.tolist()
            top5 = scores.topk(min(5, scores.size(0))).indices.tolist()
            true_id = i
            if top1 == true_id:
                recall_at_1 += 1
            if true_id in top3:
                recall_at_3 += 1
            if true_id in top5:
                recall_at_5 += 1
            total += 1

    return {
        "Recall@1": recall_at_1 / total * 100,
        "Recall@3": recall_at_3 / total * 100,
        "Recall@5": recall_at_5 / total * 100
    }

# =========================================================
# Compare raw vs denoised embeddings
# =========================================================

raw_recalls = evaluate(val_loader, use_denoiser=False)
denoised_recalls = evaluate(val_loader, use_denoiser=True)

print("📌 CLIP embeddings (raw):", raw_recalls)
print("📌 Residual denoised embeddings (1D U-Net):", denoised_recalls)

# =========================================================
#  Plot embedding similarity (cosine)
# =========================================================

def compute_alignment(img_emb, txt_emb):
    sims = F.cosine_similarity(img_emb, txt_emb)
    return sims.mean().item()

images, captions = next(iter(val_loader))
raw_img, raw_txt = get_raw_embeddings(images, captions)
denoised_img, denoised_txt = get_denoised_embeddings(images, captions)

print(f"Raw embeddings alignment (cosine): {compute_alignment(raw_img, raw_txt):.4f}")
print(f"Denoised embeddings alignment (cosine): {compute_alignment(denoised_img, denoised_txt):.4f}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/463 [00:00<?, ?B/s]

data/train-00000-of-00001-3e9e72b53b09ab(…):   0%|          | 0.00/481M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3100 [00:00<?, ? examples/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Epoch 1/10, Avg Loss: 0.081970
Epoch 2/10, Avg Loss: 0.079257
Epoch 3/10, Avg Loss: 0.077251
Epoch 4/10, Avg Loss: 0.074991
Epoch 5/10, Avg Loss: 0.071454
Epoch 6/10, Avg Loss: 0.068005
Epoch 7/10, Avg Loss: 0.064363
Epoch 8/10, Avg Loss: 0.061403
Epoch 9/10, Avg Loss: 0.058000
Epoch 10/10, Avg Loss: 0.054867
✅ Residual denoiser training complete!
📌 CLIP embeddings (raw): {'Recall@1': 40.94094094094094, 'Recall@3': 66.46646646646647, 'Recall@5': 76.87687687687688}
📌 Residual denoised embeddings (1D U-Net): {'Recall@1': 39.63963963963964, 'Recall@3': 64.46446446446447, 'Recall@5': 75.27527527527528}
Raw embeddings alignment (cosine): 0.2973
Denoised embeddings alignment (cosine): 0.4026


In [ ]:
# =========================================================
# Research-ready: Cross-modal CLIP Embedding Denoiser
# =========================================================


# =========================================================
# 1️⃣ 1D U-Net Residual Denoiser with Cross-Modal Conditioning
# =========================================================

class UNet1DResidualDenoiser(nn.Module):
    """
    1D Residual Denoiser for CLIP embeddings
    Cross-modal conditioning included
    """
    def __init__(self, embedding_dim=512, hidden_dim=1024):
        super().__init__()

        # Encoder
        self.enc1 = nn.Linear(embedding_dim * 2, hidden_dim)  # *2 for conditioning
        self.enc2 = nn.Linear(hidden_dim, hidden_dim)

        # Decoder
        self.dec1 = nn.Linear(hidden_dim, hidden_dim)
        self.dec2 = nn.Linear(hidden_dim, embedding_dim)

        self.act = nn.ReLU()

    def forward(self, x_noisy, x_clean=None, cond=None):
        """
        x_noisy: noisy embedding to denoise
        x_clean: optional clean embedding (for residual)
        cond: conditional embedding (cross-modal)
        """
        # Compute residual if clean available (training)
        if x_clean is not None:
            residual = x_noisy - x_clean
        else:
            residual = x_noisy

        # Cross-modal conditioning
        if cond is not None:
            x = torch.cat([residual, cond], dim=-1)
        else:
            x = residual

        # Encoder
        e1 = self.act(self.enc1(x))
        e2 = self.act(self.enc2(e1))

        # Decoder with skip connection
        d1 = self.act(self.dec1(e2) + e1)
        d2 = self.dec2(d1)

        # Residual subtraction
        if x_clean is not None:
            x_denoised = x_noisy - d2
        else:
            x_denoised = x_noisy - d2  # inference
        return x_denoised

# =========================================================
# 2️⃣ Noise schedule helper
# =========================================================

def add_noise_schedule(embeddings, t, max_std=0.3):
    """
    Linear noise schedule: small noise at t=0, max_std at t=1
    """
    noise_std = t * max_std
    return embeddings + torch.randn_like(embeddings) * noise_std

# =========================================================
# 3️⃣ Load small dataset
# =========================================================

dataset = load_dataset("SKyu/my-image-captioning-dataset")
train_dataset = dataset["train"].select(range(2000, 3000))
val_dataset = dataset["train"].select(range(1, 1000))

def transformFn(batch):
    images = [item["image"] for item in batch]
    captions = [item["prompt"] for item in batch]
    return images, captions

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=transformFn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=transformFn)

# =========================================================
# 4️⃣ Load frozen CLIP
# =========================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model = AutoModelForZeroShotImageClassification.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)
clip_model.eval()
for param in clip_model.parameters():
    param.requires_grad = False

# =========================================================
# 5️⃣ Initialize denoiser
# =========================================================

denoiser = UNet1DResidualDenoiser(embedding_dim=512, hidden_dim=1024).to(device)
optimizer = torch.optim.Adam(denoiser.parameters(), lr=1e-4)
criterion = nn.MSELoss()

# =========================================================
# 6️⃣ Helper functions for embeddings
# =========================================================

@torch.no_grad()
def get_clip_embeddings(images, captions):
    inputs = processor(text=captions, images=images, return_tensors="pt",
                       padding=True, truncation=True, max_length=77).to(device)
    img_embeds = F.normalize(clip_model.get_image_features(inputs["pixel_values"]), dim=-1)
    text_inputs = {k: v for k, v in inputs.items() if k in ["input_ids", "attention_mask"]}
    txt_embeds = F.normalize(clip_model.get_text_features(**text_inputs), dim=-1)
    return img_embeds, txt_embeds

def get_denoised_embeddings(images, captions):
    denoiser.eval()
    img_embeds, txt_embeds = get_clip_embeddings(images, captions)
    denoised_img = F.normalize(denoiser(img_embeds, cond=txt_embeds), dim=-1)
    denoised_txt = F.normalize(denoiser(txt_embeds, cond=img_embeds), dim=-1)
    return denoised_img, denoised_txt

# =========================================================
# 7️⃣ Training loop with cross-modal conditioning & noise schedule
# =========================================================

num_epochs = 10

for epoch in range(num_epochs):
    denoiser.train()
    epoch_loss = 0
    for images, captions in train_loader:

        # Get embeddings
        img_embeds, txt_embeds = get_clip_embeddings(images, captions)

        # Sample noise timestep t in [0,1]
        t = torch.rand(1).item()

        # Add noise
        noisy_img = add_noise_schedule(img_embeds, t)
        noisy_txt = add_noise_schedule(txt_embeds, t)

        # Denoiser forward (cross-modal conditioning)
        denoised_img = denoiser(noisy_img, x_clean=img_embeds, cond=txt_embeds)
        denoised_txt = denoiser(noisy_txt, x_clean=txt_embeds, cond=img_embeds)

        # Loss
        loss = criterion(denoised_img, img_embeds) + criterion(denoised_txt, txt_embeds)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Avg Loss: {epoch_loss/len(train_loader):.6f}")

print("✅ Cross-modal residual denoiser trained!")

# =========================================================
# 8️⃣ Evaluation: Recall@K + cosine alignment
# =========================================================

def evaluate(loader, use_denoiser=False):
    recall_at_1, recall_at_3, recall_at_5, total = 0,0,0,0
    cos_list = []

    for images, captions in loader:
        if use_denoiser:
            img_embeds, txt_embeds = get_denoised_embeddings(images, captions)
        else:
            img_embeds, txt_embeds = get_clip_embeddings(images, captions)

        sim_matrix = img_embeds @ txt_embeds.T  # [B,B]

        # Compute recalls
        for i in range(sim_matrix.size(0)):
            scores = sim_matrix[i]
            top1 = scores.topk(1).indices.item()
            top3 = scores.topk(3).indices.tolist()
            top5 = scores.topk(5).indices.tolist()
            true_id = i
            if top1 == true_id: recall_at_1 += 1
            if true_id in top3: recall_at_3 += 1
            if true_id in top5: recall_at_5 += 1
            total += 1

        # Cosine alignment
        cos_list.append(F.cosine_similarity(img_embeds, txt_embeds, dim=-1).mean().item())

    return {
        "Recall@1": recall_at_1/total*100,
        "Recall@3": recall_at_3/total*100,
        "Recall@5": recall_at_5/total*100,
        "CosineMean": sum(cos_list)/len(cos_list)
    }

raw_metrics = evaluate(val_loader, use_denoiser=False)
denoised_metrics = evaluate(val_loader, use_denoiser=True)

print("📌 Raw CLIP:", raw_metrics)
print("📌 Denoised CLIP:", denoised_metrics)

# =========================================================
# 9️⃣ Optional: Noise robustness test
# =========================================================

def robustness_test(loader, noise_levels=[0.0,0.1,0.2,0.3,0.4]):
    results = {}
    for nl in noise_levels:
        cos_list = []
        for images, captions in loader:
            img_embeds, txt_embeds = get_clip_embeddings(images, captions)
            noisy_img = add_noise_schedule(img_embeds, t=nl)
            noisy_txt = add_noise_schedule(txt_embeds, t=nl)
            denoised_img = denoiser(noisy_img, cond=txt_embeds)
            denoised_txt = denoiser(noisy_txt, cond=img_embeds)
            cos_list.append(F.cosine_similarity(denoised_img, denoised_txt, dim=-1).mean().item())
        results[nl] = sum(cos_list)/len(cos_list)
    return results

robust_results = robustness_test(val_loader)
print("Noise robustness (cosine):", robust_results)


Epoch 1/10, Avg Loss: 0.056242
Epoch 2/10, Avg Loss: 0.052573
Epoch 3/10, Avg Loss: 0.073052
Epoch 4/10, Avg Loss: 0.052898
Epoch 5/10, Avg Loss: 0.064507
Epoch 6/10, Avg Loss: 0.060316
Epoch 7/10, Avg Loss: 0.048330
Epoch 8/10, Avg Loss: 0.043823
Epoch 9/10, Avg Loss: 0.042435
Epoch 10/10, Avg Loss: 0.053699
✅ Cross-modal residual denoiser trained!
📌 Raw CLIP: {'Recall@1': 40.94094094094094, 'Recall@3': 66.46646646646647, 'Recall@5': 76.87687687687688, 'CosineMean': 0.2881801128387451}
📌 Denoised CLIP: {'Recall@1': 30.83083083083083, 'Recall@3': 57.75775775775776, 'Recall@5': 68.86886886886887, 'CosineMean': 0.45101031102240086}
Noise robustness (cosine): {0.0: 0.45101029239594936, 0.1: 0.34123914409428835, 0.2: 0.19668226642534137, 0.3: 0.11474024527706206, 0.4: 0.0728874575579539}


In [5]:
# =========================================================
# Cross-modal CLIP Residual Denoiser with MSE + Contrastive Loss
# =========================================================


# =========================================================
# 1️⃣ Define Cross-modal 1D U-Net Residual Denoiser
# =========================================================

class UNet1DResidualDenoiser(nn.Module):
    """
    1D Residual Denoiser for CLIP embeddings
    Cross-modal conditioning included
    """
    def __init__(self, embedding_dim=512, hidden_dim=1024):
        super().__init__()
        # Encoder
        self.enc1 = nn.Linear(embedding_dim*2, hidden_dim)
        self.enc2 = nn.Linear(hidden_dim, hidden_dim)
        # Decoder
        self.dec1 = nn.Linear(hidden_dim, hidden_dim)
        self.dec2 = nn.Linear(hidden_dim, embedding_dim)
        self.act = nn.ReLU()

    def forward(self, x_noisy, x_clean=None, cond=None):
        # Residual if clean embedding is available
        if x_clean is not None:
            residual = x_noisy - x_clean
        else:
            residual = x_noisy

        # Cross-modal conditioning
        if cond is not None:
            x = torch.cat([residual, cond], dim=-1)
        else:
            x = residual

        # Encoder
        e1 = self.act(self.enc1(x))
        e2 = self.act(self.enc2(e1))

        # Decoder with skip connection
        d1 = self.act(self.dec1(e2) + e1)
        d2 = self.dec2(d1)

        # Residual subtraction
        if x_clean is not None:
            x_denoised = x_noisy - d2
        else:
            x_denoised = x_noisy - d2
        return x_denoised

# =========================================================
# 2️⃣ Noise schedule helper
# =========================================================

def add_noise_schedule(embeddings, t, max_std=0.3):
    noise_std = t * max_std
    return embeddings + torch.randn_like(embeddings) * noise_std

# =========================================================
# 3️⃣ Load dataset
# =========================================================

dataset = load_dataset("SKyu/my-image-captioning-dataset")
train_dataset = dataset["train"].select(range(2000, 3000))
val_dataset = dataset["train"].select(range(1, 1000))

def transformFn(batch):
    images = [item["image"] for item in batch]
    captions = [item["prompt"] for item in batch]
    return images, captions

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=transformFn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=transformFn)

# =========================================================
# 4️⃣ Load frozen CLIP
# =========================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
processor = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model = AutoModelForZeroShotImageClassification.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)
clip_model.eval()
for param in clip_model.parameters():
    param.requires_grad = False

# =========================================================
# 5️⃣ Initialize denoiser, optimizer, criterion
# =========================================================

denoiser = UNet1DResidualDenoiser(embedding_dim=512, hidden_dim=1024).to(device)
optimizer = torch.optim.Adam(denoiser.parameters(), lr=1e-4)
criterion = nn.MSELoss()

# =========================================================
# 6️⃣ Contrastive Loss Function
# =========================================================

def contrastive_loss(img_embeds, txt_embeds, temperature=0.07):
    img_embeds = F.normalize(img_embeds, dim=-1)
    txt_embeds = F.normalize(txt_embeds, dim=-1)
    logits = img_embeds @ txt_embeds.T / temperature
    labels = torch.arange(img_embeds.size(0), device=img_embeds.device)
    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.T, labels)
    return (loss_i2t + loss_t2i) / 2

# =========================================================
# 7️⃣ Helper functions for embeddings
# =========================================================

@torch.no_grad()
def get_clip_embeddings(images, captions):
    inputs = processor(text=captions, images=images, return_tensors="pt",
                       padding=True, truncation=True, max_length=77).to(device)
    img_embeds = F.normalize(clip_model.get_image_features(inputs["pixel_values"]), dim=-1)
    text_inputs = {k: v for k, v in inputs.items() if k in ["input_ids", "attention_mask"]}
    txt_embeds = F.normalize(clip_model.get_text_features(**text_inputs), dim=-1)
    return img_embeds, txt_embeds

def get_denoised_embeddings(images, captions):
    denoiser.eval()
    img_embeds, txt_embeds = get_clip_embeddings(images, captions)
    denoised_img = F.normalize(denoiser(img_embeds, cond=txt_embeds), dim=-1)
    denoised_txt = F.normalize(denoiser(txt_embeds, cond=img_embeds), dim=-1)
    return denoised_img, denoised_txt

# =========================================================
# 8️⃣ Training loop with MSE + Contrastive Loss
# =========================================================

num_epochs = 10
for epoch in range(num_epochs):
    denoiser.train()
    epoch_loss = 0
    for images, captions in train_loader:
        img_embeds, txt_embeds = get_clip_embeddings(images, captions)
        t = torch.rand(1).item()
        noisy_img = add_noise_schedule(img_embeds, t)
        noisy_txt = add_noise_schedule(txt_embeds, t)

        # Denoising with cross-modal conditioning
        denoised_img = denoiser(noisy_img, x_clean=img_embeds, cond=txt_embeds)
        denoised_txt = denoiser(noisy_txt, x_clean=txt_embeds, cond=img_embeds)

        # Compute combined loss
        mse_loss = criterion(denoised_img, img_embeds) + criterion(denoised_txt, txt_embeds)
        cont_loss = contrastive_loss(denoised_img, denoised_txt, temperature=0.07)
        loss = mse_loss + cont_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1}/{num_epochs}, Avg Loss: {epoch_loss/len(train_loader):.6f}")

print("✅ Cross-modal residual denoiser trained with contrastive loss!")

# =========================================================
# 9️⃣ Evaluation: Recall@K + Cosine Alignment
# =========================================================

def evaluate(loader, use_denoiser=False):
    recall_at_1, recall_at_3, recall_at_5, total = 0, 0, 0, 0
    cos_list = []
    for images, captions in loader:
        if use_denoiser:
            img_embeds, txt_embeds = get_denoised_embeddings(images, captions)
        else:
            img_embeds, txt_embeds = get_clip_embeddings(images, captions)
        sim_matrix = img_embeds @ txt_embeds.T
        for i in range(sim_matrix.size(0)):
            scores = sim_matrix[i]
            top1 = scores.topk(1).indices.item()
            top3 = scores.topk(3).indices.tolist()
            top5 = scores.topk(5).indices.tolist()
            true_id = i
            if top1 == true_id: recall_at_1 += 1
            if true_id in top3: recall_at_3 += 1
            if true_id in top5: recall_at_5 += 1
            total += 1
        cos_list.append(F.cosine_similarity(img_embeds, txt_embeds, dim=-1).mean().item())
    return {
        "Recall@1": recall_at_1/total*100,
        "Recall@3": recall_at_3/total*100,
        "Recall@5": recall_at_5/total*100,
        "CosineMean": sum(cos_list)/len(cos_list)
    }

raw_metrics = evaluate(val_loader, use_denoiser=False)
denoised_metrics = evaluate(val_loader, use_denoiser=True)
print("📌 Raw CLIP:", raw_metrics)
print("📌 Denoised CLIP:", denoised_metrics)

# =========================================================
# 🔟 Optional: Noise robustness test
# =========================================================

def robustness_test(loader, noise_levels=[0.0,0.1,0.2,0.3,0.4]):
    results = {}
    for nl in noise_levels:
        cos_list = []
        for images, captions in loader:
            img_embeds, txt_embeds = get_clip_embeddings(images, captions)
            noisy_img = add_noise_schedule(img_embeds, t=nl)
            noisy_txt = add_noise_schedule(txt_embeds, t=nl)
            denoised_img = denoiser(noisy_img, cond=txt_embeds)
            denoised_txt = denoiser(noisy_txt, cond=img_embeds)
            cos_list.append(F.cosine_similarity(denoised_img, denoised_txt, dim=-1).mean().item())
        results[nl] = sum(cos_list)/len(cos_list)
    return results

robust_results = robustness_test(val_loader)
print("Noise robustness (cosine):", robust_results)


Epoch 1/10, Avg Loss: 3.521063
Epoch 2/10, Avg Loss: 3.397909
Epoch 3/10, Avg Loss: 3.095990
Epoch 4/10, Avg Loss: 3.063170
Epoch 5/10, Avg Loss: 3.103211
Epoch 6/10, Avg Loss: 2.824786
Epoch 7/10, Avg Loss: 2.432263
Epoch 8/10, Avg Loss: 2.530532
Epoch 9/10, Avg Loss: 2.113903
Epoch 10/10, Avg Loss: 1.935422
✅ Cross-modal residual denoiser trained with contrastive loss!
📌 Raw CLIP: {'Recall@1': 40.94094094094094, 'Recall@3': 66.46646646646647, 'Recall@5': 76.87687687687688, 'CosineMean': 0.2881801128387451}
📌 Denoised CLIP: {'Recall@1': 46.94694694694695, 'Recall@3': 75.27527527527528, 'Recall@5': 85.98598598598599, 'CosineMean': 0.8134176284074783}
Noise robustness (cosine): {0.0: 0.8134175837039948, 0.1: 0.7394041530787945, 0.2: 0.5794345522299409, 0.3: 0.42854904755949974, 0.4: 0.31327595841139555}
